In [1]:
import opt_einsum as oe
import numpy as np
import torch
import sys
sys.path.append("../../")
import mps
from mps.trainer.data_utils import create_mnist_dataloader

In [9]:
N = 10
dataset = create_mnist_dataloader(N, batch_size=2**10, allowed_digits=[3, 5])
dataloader = torch.utils.data.DataLoader(dataset, batch_size=2**10, shuffle=True)

In [10]:
from mps.trainer.smps_trainer import smps_train
from mps.simple_mps import SimpleMPS
import copy
# smps_params = torch.load("smps_2000.pth", weights_only=False)

chi = 8
d = 2
l = 2
device = torch.device("cpu")
dtype = torch.float64
optimize = "greedy"
eps = 1 / np.sqrt(N) * 0.1
smps = SimpleMPS(N, chi, d, l, layers=1, device=device, dtype=dtype, optimize=optimize, eps=eps)

# smps4.mps.set_params(smps_params)
# smps4.initialize_MPS()
# smps = smps4.compress_chi(2)

Path is not set, setting...
Found the path
Initialized MPS with random matrices


In [11]:
from mps.trainer.smps_trainer import smps_train
import copy

epochs = 10
lr = 0.00001
logsoftmax = torch.nn.LogSoftmax(dim=-1)
nnloss = torch.nn.NLLLoss(reduction="mean")
opt_smps = torch.optim.Adam(smps.parameters(), lr=lr)
smps_losses = []
smps.train()
print(f"\n=== Training SimpleMPS for {epochs} epoch(s)... ===")
for epoch in range(epochs):
    total_loss = 0.0
    total_samples = 0
    total_correct = 0
    for batch_idx, (data, target) in enumerate(dataloader):
        data, target = data.to(device), target.to(device)
        data = data.permute(1, 0, 2)  # [batch, N, 2] → [N, batch, 2]
        opt_smps.zero_grad()
        outputs = smps(data)
        outputs = torch.abs(outputs)
        outputs = logsoftmax(outputs)
        loss = nnloss(outputs, target)
        loss.backward()
        
        # Print the norm of the gradients for 10 equally split indices
        params = list(smps.parameters())
        num_params = len(params)
        indices = [int(i * num_params / 10) for i in range(10)]
        grad_norms = [f"Gradient norm for parameter {idx}: {params[idx].grad.norm().item():.6f}" for idx in indices if params[idx].grad is not None]
        print(" | ".join(grad_norms))
        
        opt_smps.step()
        bs = target.size(0)
        total_loss += loss.item() * bs
        total_samples += bs
        preds = outputs.argmax(dim=-1)
        acc = (preds == target).float().sum().item()
        total_correct += acc
        print(f"[SimpleMPS] Epoch {epoch+1}, Step {batch_idx+1}/{len(dataloader)} | Loss: {loss.item():.6f} | Acc: {acc/bs:.2%}")
    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples
    smps_losses.append(epoch_loss)
    print(f"[SimpleMPS] Epoch {epoch+1} | Loss: {epoch_loss:.6f} | Acc: {epoch_acc:.2%}")


=== Training SimpleMPS for 10 epoch(s)... ===
Gradient norm for parameter 0: 0.000688 | Gradient norm for parameter 1: 0.000291 | Gradient norm for parameter 2: 0.001216 | Gradient norm for parameter 3: 0.000390 | Gradient norm for parameter 4: 0.000248 | Gradient norm for parameter 5: 0.000942 | Gradient norm for parameter 6: 0.000991 | Gradient norm for parameter 7: 0.001160 | Gradient norm for parameter 8: 0.000178 | Gradient norm for parameter 9: 0.001099
[SimpleMPS] Epoch 1, Step 1/32768 | Loss: 0.693238 | Acc: 48.05%
Gradient norm for parameter 0: 0.001861 | Gradient norm for parameter 1: 0.002520 | Gradient norm for parameter 2: 0.001844 | Gradient norm for parameter 3: 0.002109 | Gradient norm for parameter 4: 0.001670 | Gradient norm for parameter 5: 0.001538 | Gradient norm for parameter 6: 0.001670 | Gradient norm for parameter 7: 0.001729 | Gradient norm for parameter 8: 0.001461 | Gradient norm for parameter 9: 0.001897
[SimpleMPS] Epoch 1, Step 2/32768 | Loss: 0.692950 |

KeyboardInterrupt: 

In [3]:
import opt_einsum as oe
import numpy as np
import torch
import sys
sys.path.append("../../")
import mps
from mps import tpcp_mps
from mps.simple_mps import SimpleMPS
from mps.trainer.data_utils import SyntheticDataset, SyntheticDatasetV2, SyntheticDatasetV3
smps_params = torch.load("smps_2000_2.pth", weights_only=False)

N = 2000
dataset = SyntheticDatasetV3(n=N, num_samples=(2**25), seed=42)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=2**10, shuffle=True)

chi = 2
d = 2
l = 2
device = torch.device("cpu")
dtype = torch.float64
optimize = "greedy"
eps = 1e-2
smps = SimpleMPS(N, chi, d, l, layers=1, device=device, dtype=dtype, optimize=optimize, eps=eps)

smps.mps.set_params(smps_params)
smps.initialize_MPS()

Path is not set, setting...
Found the path
Initialized MPS with random matrices
Initialized MPS with random matrices


In [4]:
from mps import tpcp_mps
from mps.trainer.utils import calculate_accuracy, loss_batch
# --- Step 2: Build and Prepare TPCP ---
tpcp = tpcp_mps.MPSTPCP(
    N,
    K=1,
    d=2,
    enable_r=True,
    with_identity=False,
    manifold=tpcp_mps.ManifoldType.EXACT,
)
tpcp.train()
tpcp.set_canonical_mps(smps)

logsoftmax = torch.nn.LogSoftmax(dim=-1)
nnloss = torch.nn.NLLLoss(reduction="mean")

# Initialize W: start with first column ones and second column small.

In [5]:
W_now = tpcp.W.clone()

def update_weights(W, rate):
    max_W = torch.max(W, dim=1, keepdim=True).values
    W_update = W.clone()
    diff = max_W - W
    with torch.no_grad():
        W_update.data.add_(rate * diff)
    return W_update

In [6]:
W = torch.zeros(tpcp.L, 2, dtype=torch.float64)
W[:, 0] = 1 
W[:, 1] = 0.001
tpcp.initialize_W(W)
# W_now = tpcp.W.clone()
# W_new = update_weights(W_now, 0.001)
# tpcp.initialize_W(W_new)
# 
# --- Step 3: Determine lambda_final Using the Initial Loss Value ---
data_batch, target_batch = next(iter(dataloader))
initial_probs, reg = tpcp(data_batch, return_probs=True, return_reg=True)
# softmax_initial_probs = logsoftmax(initial_probs)
initial_accuracy = calculate_accuracy(initial_probs[:, 0], target_batch)
print(f"Initial accuracy: {initial_accuracy.item():.2%}")

loss = loss_batch(initial_probs[:, 0], target_batch)
print(f"Initial loss: {loss.item()}")

Initial accuracy: 41.60%
Initial loss: 0.7514698386878338


In [7]:
# from mps.trainer.adaptive_mpsae_trainer import RiemannianAdam
from geoopt import optim
from mps.StiefelOptimizers import StiefelAdam, StiefelSGD
import torch
lr = 0.00001
# optimizer = StiefelAdam(tpcp.kraus_ops.parameters(), lr=lr, expm_method='Cayley')
optimizer = optim.RiemannianAdam(tpcp.kraus_ops.parameters(), lr=lr, betas=(0.99, 0.999))
optimizer_weight = torch.optim.Adam([tpcp.r, tpcp.W], lr=lr)
tpcp.W.requires_grad = True
tpcp.r.requires_grad = True

In [8]:
clambda = 300
epochs = 1000
for _ in range(epochs):
    for data, target in dataloader:
        optimizer.zero_grad()
        optimizer_weight.zero_grad()
        outputs, reg = tpcp(data, return_probs=True, return_reg=True)
        # probs = logsoftmax(outputs)
        # loss0 = nnloss(probs, target)
        loss0 = loss_batch(outputs[:, 0], target)
        loss = loss0  + clambda * reg

        loss.backward()

        optimizer.step()
        optimizer_weight.step()

        tpcp.proj_stiefel(check_on_manifold=True, print_log=False, rtol=1e-3)
        tpcp.normalize_w_and_r()

        acc = calculate_accuracy(outputs[:, 0], target)
        srpq = torch.exp(-reg)
        print(f"Loss0 : {loss0.item():.6f}, reg: {reg.item():.6f}, Loss: {loss.item():.6f}, Acc: {acc.item():.2%}, SRPQ: {srpq.item():.6e}")


Loss0 : 0.741933, reg: 0.692234, Loss: 208.412072, Acc: 43.55%, SRPQ: 5.004569e-01
Loss0 : 0.740771, reg: 0.692216, Loss: 208.405547, Acc: 43.65%, SRPQ: 5.004658e-01
Loss0 : 0.741602, reg: 0.692217, Loss: 208.406620, Acc: 43.95%, SRPQ: 5.004654e-01
Loss0 : 0.723384, reg: 0.692195, Loss: 208.381946, Acc: 48.83%, SRPQ: 5.004762e-01
Loss0 : 0.720855, reg: 0.692171, Loss: 208.372238, Acc: 48.73%, SRPQ: 5.004882e-01
Loss0 : 0.716633, reg: 0.692178, Loss: 208.370008, Acc: 49.90%, SRPQ: 5.004849e-01
Loss0 : 0.714561, reg: 0.692160, Loss: 208.362471, Acc: 49.22%, SRPQ: 5.004940e-01
Loss0 : 0.690372, reg: 0.692158, Loss: 208.337818, Acc: 54.30%, SRPQ: 5.004948e-01
Loss0 : 0.707385, reg: 0.692164, Loss: 208.356704, Acc: 49.71%, SRPQ: 5.004916e-01
Loss0 : 0.696216, reg: 0.692143, Loss: 208.339072, Acc: 53.32%, SRPQ: 5.005024e-01
Loss0 : 0.686442, reg: 0.692133, Loss: 208.326312, Acc: 55.08%, SRPQ: 5.005074e-01
Loss0 : 0.679518, reg: 0.692109, Loss: 208.312233, Acc: 56.54%, SRPQ: 5.005193e-01
Loss

: 

In [136]:
i = 1
x = data_batch[i]
num_K = tpcp.K

rho0 = torch.einsum("i, j->ij", x[0], x[0].conj())
rho1 = torch.einsum("i, j->ij", x[1], x[1].conj())
rho_in = torch.einsum("ij,kl->ikjl", rho0, rho1).reshape(4, 4)
K = tpcp.kraus_ops[0].data.reshape(num_K, 4, 4)
rho_k_out = torch.einsum("kij, jl, kml->im", K, rho_in, K.conj()).reshape(2, 2, 2, 2)
#partial trace
rho_k_out = torch.einsum("ijil -> jl", rho_k_out)

for k in range(1, len(tpcp.kraus_ops.kraus_ops)):
    K = tpcp.kraus_ops[k].data.reshape(num_K, 4, 4)
    rho_new = torch.einsum("i, j->ij", x[k+1], x[k+1].conj())
    rho_k_in = torch.einsum("ij,kl->ikjl", rho_k_out, rho_new).reshape(4, 4)
    rho_k_out = torch.einsum("kij, jl, kml->im", K, rho_k_in, K.conj()).reshape(2, 2, 2, 2)
    rho_last = rho_k_out.data.reshape(4, 4)
    #partial trace
    rho_k_out = torch.einsum("ijil -> jl", rho_k_out)

mes0 = tpcp.r.conj() @ tpcp.pros0 @ tpcp.r.T - tpcp.r.conj() @ tpcp.pros1 @ tpcp.r.T

print(torch.trace(mes0 @ rho_k_out), target_batch[i])
print(tpcp(data_batch[i].unsqueeze(0)), target_batch[i])

tensor(-0.4681, dtype=torch.float64, grad_fn=<TraceBackward0>) tensor(1)
tensor([0.0036], dtype=torch.float64, grad_fn=<SelectBackward0>) tensor(1)


In [452]:
tpcp(data_batch[i].unsqueeze(0)), target_batch[i]

(tensor([0.9907], dtype=torch.float64, grad_fn=<SelectBackward0>), tensor(0))

In [59]:
MPS_list[0].shape

(2, 4)

In [24]:
d = smps.mps.d
chi = smps.mps.chi_max
MPS_list = [core.detach().cpu().numpy() for core in smps.mps.params]
MPS_list[0] = MPS_list[0].reshape(1, d, chi)
d = smps.mps.d


new_MPS_list = [np.zeros(mps_shape.shape) for mps_shape in MPS_list]

A = np.copy(MPS_list[0])

for i in range(len(MPS_list) - 1):

    shapeA = A.shape
    A = A.reshape(A.shape[0] * d, -1)
    
    # Perform SVD
    U, S, V = np.linalg.svd(A, full_matrices=False)

    S = S / np.linalg.norm(S)

    S[4:] = 0 
    U[:, 4:] = 0
    
    # Update chi and R
    chi = len(S)
    R = np.diag(S) @ V[:chi]

    if i < len(MPS_list) - 2:
        A = np.copy(MPS_list[i+1])
        A = np.einsum("ij, jdk->idk", R, A)

    # new_MPS_list.append(U.reshape(shapeA[0], shapeA[1], -1))

    At = U.reshape(shapeA[0], shapeA[1], -1)
    Atshape = At.shape
    new_MPS_list[i][:Atshape[0], :Atshape[1], :Atshape[2]] = At

Af = np.copy(MPS_list[-1])
Af = R @ Af
new_MPS_list[-1][:] = Af

new_MPS_list[0] = new_MPS_list[0].reshape(d, smps.mps.chi_max)
new_MPS_list_tensor = [torch.tensor(new_MPS_list[i]) for i in range(len(new_MPS_list))]

# smps.mps.set_params(new_MPS_list_tensor)
# smps.initialize_MPS()

In [34]:
params = smps.mps.params

chi = 2
smps_new = SimpleMPS(N, chi, d, l, layers=1, device=device, dtype=dtype, optimize=optimize, eps=eps)

params_new = [torch.empty(shape) for shape in smps_new.mps.mps_shapes]

smps.convert_to_canonical()

Path is not set, setting...
Found the path
Initialized MPS with random matrices


AttributeError: 'SimpleMPS' object has no attribute 'convert_to_canonical'

In [52]:
smps_params = torch.load("smps.pth", weights_only=False)

chi = 4
d = 2
l = 2
device = torch.device("cpu")
dtype = torch.float64
optimize = "greedy"
eps = 1e-2
smps = SimpleMPS(N, chi, d, l, layers=1, device=device, dtype=dtype, optimize=optimize, eps=eps)

smps.mps.set_params(smps_params)
smps.initialize_MPS()

d = smps.mps.d
chi = smps.mps.chi_max
MPS_list = [core.detach().cpu().numpy() for core in smps.mps.params]
MPS_list[0] = MPS_list[0].reshape(1, d, chi)
d = smps.mps.d


new_MPS_list = [np.zeros(mps_shape.shape) for mps_shape in MPS_list]

A = np.copy(MPS_list[0])

for i in range(len(MPS_list) - 1):

    shapeA = A.shape
    A = A.reshape(A.shape[0] * d, -1)
    
    # Perform SVD
    U, S, V = np.linalg.svd(A, full_matrices=False)

    U[:, 2:] = 0
    S[2:] = 0
    V[2:, :] = 0
    S = S / np.linalg.norm(S)

    
    # Update chi and R
    chi = len(S)
    R = np.diag(S) @ V[:chi]

    if i < len(MPS_list) - 2:
        A = np.copy(MPS_list[i+1])
        A = np.einsum("ij, jdk->idk", R, A)

    # new_MPS_list.append(U.reshape(shapeA[0], shapeA[1], -1))

    At = U.reshape(shapeA[0], shapeA[1], -1)
    Atshape = At.shape
    new_MPS_list[i][:Atshape[0], :Atshape[1], :Atshape[2]] = At

Af = np.copy(MPS_list[-1])
Af = R @ Af
new_MPS_list[-1][:] = Af

new_MPS_list[0] = new_MPS_list[0].reshape(d, smps.mps.chi_max)
new_MPS_list_tensor = [torch.tensor(new_MPS_list[i]) for i in range(len(new_MPS_list))]

smps.mps.set_params(new_MPS_list_tensor)
smps.initialize_MPS()


Path is not set, setting...
Found the path
Initialized MPS with random matrices
Initialized MPS with random matrices
Initialized MPS with random matrices


In [54]:
new_MPS_list[8][:2, :, :2]

array([[[-6.96452678e-01,  6.11388782e-03],
        [-7.17602633e-01, -6.20796586e-03]],

       [[ 3.50330523e-04, -7.09033352e-01],
        [-7.31461955e-05, -7.05121115e-01]]])

In [261]:
# Try to normalize the MPS
# FIrst prepare the test state
x = torch.randn(N, 2, dtype=smps.mps.dtype, device=smps.mps.device)
# x[:, 0] = 1
x = torch.abs(x)
x = x / x.sum(dim=1).unsqueeze(1)

out = smps(x.reshape(N, 1, 2))
sys_scale = torch.abs(out).sum()
s = sys_scale ** (1 / N)

print(sys_scale)

params = smps.mps.params
params_t = [p.detach().cpu() for p in params]
for i in range(len(params)):
   params_t[i] = params_t[i] * (1 / s)# 

smps.mps.set_params(params_t)
smps.initialize_MPS()

# Check the scale again
out = smps(x.reshape(N, 1, 2))
sys_scale = torch.abs(out).sum()

sys_scale




tensor(7.7383e-77, dtype=torch.float64, grad_fn=<SumBackward0>)
Initialized MPS with random matrices


tensor(1.4198, dtype=torch.float64, grad_fn=<SumBackward0>)

In [262]:
smps(data)

tensor([[0.9485, 1.5121],
        [0.6512, 0.1270],
        [0.9966, 1.5054],
        ...,
        [1.4029, 1.7655],
        [0.6907, 0.2395],
        [0.9409, 1.4726]], dtype=torch.float64, grad_fn=<AbsBackward0>)

In [185]:
1e-78 ** (1 / N)

0.6982324040771714

In [193]:
smps(data)

tensor([[0.5329, 0.9569],
        [0.3981, 0.0426],
        [0.5706, 0.9784],
        ...,
        [0.8310, 1.1142],
        [0.4377, 0.0711],
        [0.5476, 0.9555]], dtype=torch.float64, grad_fn=<AbsBackward0>)